# Submiting to Argo
This file will let your submit to argo quicker than writing out all the commands in the terminal.

# Setup
Here we use kibectl to interact with any pods that argo uses. An example for listing all the resources under your namespce is here:

In [ ]:
%%bash
kubectl -n csa-ml-proto-argo get pod -o wide | grep dask-

## Hera (Quick Intro)
The code used to generate the yaml, uses Hera. Hera is a Python library for constructing Argo Workflows programmatically. It lets you build, parameterise, and output Argo workflow YAML from Python. In this repo the example builders live in `notebooks/dask/Argo_Files/argo_dataset_build.py` and `notebooks/dask/Argo_Files/argo_train.py` which use Hera's `@script` decorator and `Workflow` objects to generate the final YAML.

<details>
    Note: The current notebooks/dask/Argo_Files/argo_train.py fiel is untested and incomplete.
</details>

#### Send your config file to S3

In [ ]:
%%bash
aws s3 cp ~/easi-notebooks/notebooks/dask/Argo_Files/training_dataset.json s3://easihub-csiro-dc-data-projects/ml-proto/argo-test/config.json


# Create yaml file
Replace with the paths to your files

In [6]:
%%bash

python argo_dataset_build.py > build_dataset.yaml

# Submit argo
Add any flags here to overwrite the default options

In [ ]:
%%bash
BUCKET="easihub-csiro-dc-data-projects"
BASE_PREFIX="..."
FOLDER_PREFIX="..."
LOCAL_CONFIG="./config.json"
S3_CONFIG_URI="s3://${BUCKET}/${BASE_PREFIX}/${FOLDER_PREFIX}/config.json"

# submit the workflow
argo submit build_dataset.yaml \
  -p config_path="${S3_CONFIG_URI}" \
  -p bucket="${BUCKET}" \
  -p base_prefix="${BASE_PREFIX}/${FOLDER_PREFIX}" \
  -p dataset_name="training_dataset_v4" \
  -p resume="True"
#  --watch --log

# Check it worked
Should show up as `PRE <dataset-components>/`

In [91]:
%%bash
# Ignore if your env is set up correctly
unset AWS_ACCESS_KEY_ID
unset AWS_SESSION_TOKEN


aws s3 ls .../path_to/training_dataset_v4

                           PRE chunks/
                           PRE manifests/
                           PRE refs/
                           PRE snapshots/
                           PRE transactions/


If you want to delete it:

In [ ]:
%%bash

aws s3 rm "..."


# Training
Work in progress but same idea

In [ ]:
%%bash

python training_workflow.py > training.yaml

In [ ]:
%%bash
# Send model file to bucket



In [ ]:
%%bash
BUCKET="easihub-csiro-dc-data-projects"
BASE_PREFIX="ml-proto"
TEST_PREFIX="argo-test"
REPO_PREFIX="${BASE_PREFIX}/${TEST_PREFIX}"
OUTPUT_DIR="models/experiment-1"
LOG_DIR="logs/experiment-1"
MODEL_PATH="s3://${BUCKET}/${REPO_PREFIX}/user_model_def.py"
MODEL_NAME="${MODEL_NAME:-PrithviSegmentation}"


argo submit training.yaml \
  -p repo_prefix="${REPO_PREFIX}" \
  -p s3_bucket="${BUCKET}" \
  -p output_dir="${OUTPUT_DIR}" \
  -p log_dir="${LOG_DIR}" \
  -p model_path="${MODEL_PATH}" \
  -p model_name="${MODEL_NAME}" \

